<a href="https://colab.research.google.com/github/luanoliveira-pixel/pcd-aulasiniciais/blob/main/EI_III_coleta_dados_clima_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coleta automática de dados meteorológicos com a API Open-Meteo

**Disciplina:** Programação para Ciência de Dados

**Autor:** Luan

**Objetivo:** desenvolver uma aplicação em Python que consome uma API pública
(a [Open-Meteo](https://open-meteo.com/)) para coletar automaticamente dados
meteorológicos atuais de várias capitais brasileiras, estruturar esses dados
e salvá-los em um arquivo JSON para uso em atividades futuras.

Escolhi a Open-Meteo porque é uma API pública, gratuita e que não
exige cadastro nem API key, o que facilita bastante para
quem está começando a trabalhar com APIs.


## 1. Importa as bibliotecas

In [25]:
import requests
import json
from datetime import datetime

## 2. Define as cidades que vamos consultar

Pra não deixar o código gigante, guardei as coordenadas de algumas capitais
brasileiras em um dicionário. Cada cidade tem sua latitude, longitude e a
sigla do estado. Esses dados eu peguei direto do Google.

In [ ]:
# dicionário com as cidades que eu quero coletar o clima
# a chave é o nome da cidade e o valor é outro dicionário com lat, lon e uf
CIDADES = {
    "São Paulo":        {"lat": -23.55,  "lon": -46.63,  "uf": "SP"},
    "Rio de Janeiro":    {"lat": -22.91,  "lon": -43.17,  "uf": "RJ"},
    "Porto Alegre":      {"lat": -30.03,  "lon": -51.23,  "uf": "RS"},
    "Brasília":          {"lat": -15.78,  "lon": -47.93,  "uf": "DF"},
    "Salvador":          {"lat": -12.97,  "lon": -38.51,  "uf": "BA"},
    "Manaus":            {"lat": -3.12,   "lon": -60.02,  "uf": "AM"},
    "Curitiba":          {"lat": -25.43,  "lon": -49.27,  "uf": "PR"},
    "Belém":             {"lat": -1.46,   "lon": -48.50,  "uf": "PA"},
}

print(f"Vamos coletar o clima de {len(CIDADES)} cidades.")

## 3. Traduz os códigos de clima

A API não devolve "ensolarado" ou "chuvoso" direto, ela devolve um número
chamado weathercode. Então eu criei um dicionário pra traduzir esses
códigos mais comuns pra um texto que faz sentido.
A lista completa
de códigos está na [documentação da Open-Meteo](https://open-meteo.com/en/docs),
mas aqui eu só coloquei os mais frequentes pra não encher o código de coisa
que talvez nem apareça.


In [27]:
# tradução simplificada do "weathercode" da Open-Meteo (padrão WMO)
# se aparecer algum código que eu não mapeei, uso um texto genérico lá na frente
CODIGOS_TEMPO = {
    0: "Céu limpo",
    1: "Predominantemente limpo",
    2: "Parcialmente nublado",
    3: "Nublado",
    45: "Neblina",
    48: "Neblina com geada",
    51: "Garoa fraca",
    53: "Garoa moderada",
    55: "Garoa forte",
    61: "Chuva fraca",
    63: "Chuva moderada",
    65: "Chuva forte",
    71: "Neve fraca",
    80: "Pancadas de chuva fracas",
    81: "Pancadas de chuva moderadas",
    95: "Tempestade",
}

## 4. Função que consulta a API

Agora vem a parte principal: uma função que recebe a latitude e a longitude
de uma cidade e devolve o clima atual dela, já pegando só os campos que a
gente precisa (a API devolve bem mais coisa do que a gente vai usar).

Coloquei um try/except porque, na vida real, uma requisição pode falhar
(a internet pode cair, a API pode estar fora do ar, etc.) e eu não quero que
isso derrube o programa inteiro — só quero pular aquela cidade e seguir pras
próximas.


In [28]:
def buscar_clima_atual(latitude, longitude):
    """
    Consulta a API Open-Meteo e devolve um dicionário com o clima atual
    para a latitude/longitude informadas.
    Se der algum erro na requisição, devolve None (e quem chamou a função
    decide o que fazer com isso).
    """
    url = "https://api.open-meteo.com/v1/forecast"

    # parâmetros da requisição: não precisa de API key, só mandar lat/lon
    # e dizer que a gente quer o "current_weather"
    parametros = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True,
        "timezone": "auto",  # pra API já ajustar o horário pro fuso da cidade
    }

    try:
        resposta = requests.get(url, params=parametros, timeout=10)
        resposta.raise_for_status()  # se vier erro HTTP (ex: 404, 500), já dispara exceção
    except requests.exceptions.RequestException as erro:
        print(f"  -> Falha ao consultar a API: {erro}")
        return None

    dados_json = resposta.json()
    clima_atual = dados_json.get("current_weather")

    if clima_atual is None:
        print("  -> A resposta da API veio sem o campo 'current_weather'.")
        return None

    return clima_atual

## 5. Coletando os dados de todas as cidades

Aqui eu percorro o dicionário de cidades, chamo a função pra cada uma delas
e vou montando uma lista com os resultados já organizados do jeito que eu
quero guardar no JSON final.


In [ ]:
dados_coletados = []  # aqui vou guardar o resultado de cada cidade

for nome_cidade, info in CIDADES.items():
    print(f"Coletando dados de {nome_cidade}...")

    clima = buscar_clima_atual(info["lat"], info["lon"])

    if clima is None:
        print(f"  -> Não foi possível obter os dados de {nome_cidade}. Pulando.\n")
        continue

    codigo_tempo = clima.get("weathercode")

    # monto um registro só com o que interessa
    registro = {
        "cidade": nome_cidade,
        "estado": info["uf"],
        "latitude": info["lat"],
        "longitude": info["lon"],
        "temperatura_celsius": clima.get("temperature"),
        "velocidade_vento_kmh": clima.get("windspeed"),
        "direcao_vento_graus": clima.get("winddirection"),
        "condicao_tempo": CODIGOS_TEMPO.get(codigo_tempo, f"Código desconhecido ({codigo_tempo})"),
        "e_dia": bool(clima.get("is_day")),
        "horario_medicao": clima.get("time"),
    }

    dados_coletados.append(registro)
    print(f"  -> OK! {registro['temperatura_celsius']}°C, {registro['condicao_tempo']}\n")

print(f"Coleta finalizada. {len(dados_coletados)} de {len(CIDADES)} cidades coletadas com sucesso.")

## 6. Organizando tudo e salvando em JSON

Agora eu monto um dicionário com algumas informações sobre a coleta
(metadados) e a lista de cidades, e salvo isso num arquivo .json.

Usei indent=2 só pra deixar o arquivo bonitinho e fácil de ler, e
ensure_ascii=False pra acentos e "ç" aparecerem certos no arquivo, em vez
de ficarem codificados tipo \u00e7.


In [ ]:
NOME_ARQUIVO = "dados_meteorologicos.json"

estrutura_final = {
    "metadados": {
        "fonte": "Open-Meteo API (https://open-meteo.com)",
        "data_coleta": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_cidades_consultadas": len(CIDADES),
        "total_cidades_coletadas": len(dados_coletados),
    },
    "cidades": dados_coletados,
}

with open(NOME_ARQUIVO, "w", encoding="utf-8") as arquivo:
    json.dump(estrutura_final, arquivo, ensure_ascii=False, indent=2)

print(f"Arquivo '{NOME_ARQUIVO}' salvo com sucesso!")

## 7. Conferindo se deu tudo certo

Eu escolhi reabrir o arquivo que
acabei de salvar e ler os dados de volta, pra ter certeza de que ele
foi gravado corretamente e que a estrutura está do jeito que eu esperava.

Aproveitei também pra tirar uma estatística bem simples (cidade mais quente e
mais fria), só pra mostrar que dá
pra fazer análises em cima deles depois.


In [ ]:
with open(NOME_ARQUIVO, "r", encoding="utf-8") as arquivo:
    dados_lidos = json.load(arquivo)

print("Metadados da coleta:")
print(dados_lidos["metadados"])
print()

print("Cidades coletadas:")
for cidade in dados_lidos["cidades"]:
    print(f"  - {cidade['cidade']}/{cidade['estado']}: "
          f"{cidade['temperatura_celsius']}°C, {cidade['condicao_tempo']}")

# só roda a estatística se tiver coletado pelo menos uma cidade
if dados_lidos["cidades"]:
    mais_quente = max(dados_lidos["cidades"], key=lambda c: c["temperatura_celsius"])
    mais_fria = min(dados_lidos["cidades"], key=lambda c: c["temperatura_celsius"])

    print()
    print(f"Cidade mais quente agora: {mais_quente['cidade']} ({mais_quente['temperatura_celsius']}°C)")
    print(f"Cidade mais fria agora:   {mais_fria['cidade']} ({mais_fria['temperatura_celsius']}°C)")

## 8. Baixando o arquivo (opcional, só funciona no Google Colab)

Esse último bloco só serve se você estiver rodando no Google Colab: ele abre
a janela de download do navegador pra você salvar o dados_meteorologicos.json
na sua máquina.


In [32]:
try:
    from google.colab import files
    files.download(NOME_ARQUIVO)
except ImportError:
    # se não estiver no Colab (ex: rodando localmente), não tem o que baixar,
    # o arquivo já está salvo na pasta onde o código está rodando
    print("Não estou no Google Colab, então não tem o que baixar por aqui.")
    print(f"O arquivo já está salvo localmente como '{NOME_ARQUIVO}'.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>